In [4]:
import importlib
packages = ['xgboost', 'tensorflow', 'sklearn', 'ta', 'numpy', 'pandas']
for pkg in packages:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, '__version__', 'unknown')
        print(f"  ✅ {pkg:15} {ver}")
    except:
        print(f"  ❌ {pkg:15} NOT installed")

  ✅ xgboost         3.2.0
  ✅ tensorflow      2.19.0
  ✅ sklearn         1.5.1
  ✅ ta              unknown
  ✅ numpy           1.26.4
  ✅ pandas          3.0.1


In [7]:
import pandas as pd
import pickle

# Check pkl files
with open('../data/price_data.pkl', 'rb') as f:
    price_data = pickle.load(f)
    
with open('../data/indicator_data.pkl', 'rb') as f:
    indicator_data = pickle.load(f)

fund_scores = pd.read_csv('../data/fundamental_scores.csv')

print(f"Stocks in price_data    : {len(price_data)}")
print(f"Stocks in indicator_data: {len(indicator_data)}")
print(f"Stocks in fund_scores   : {len(fund_scores)}")
print(f"\nSample symbols: {list(price_data.keys())[:5]}")

Stocks in price_data    : 37
Stocks in indicator_data: 37
Stocks in fund_scores   : 37

Sample symbols: ['PERSISTENT', 'CHOLAFIN', 'MUTHOOTFIN', 'FINEORG', 'TITAN']


In [8]:
# ============================================================
# DAY 7 — ML MODELS
# AI Stock Screener — Indian Markets
# Models: Reversal Classifier, Trend Classifier, Price Forecast
# ============================================================

# ── CELL 1: IMPORTS & SETUP ──────────────────────────────────
import pandas as pd
import numpy as np
import pickle
import warnings
import os
warnings.filterwarnings('ignore')

from xgboost import XGBClassifier, XGBRegressor
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
from collections import defaultdict

# Create models directory if not exists
os.makedirs('../models', exist_ok=True)
os.makedirs('../data', exist_ok=True)

print("✅ Imports done")

✅ Imports done


In [12]:
# ── CELL 2: LOAD ALL DATA ────────────────────────────────────
with open('../data/price_data.pkl', 'rb') as f:
    price_data = pickle.load(f)

with open('../data/indicator_data.pkl', 'rb') as f:
    indicator_data = pickle.load(f)

fund_scores  = pd.read_csv('../data/fundamental_scores.csv')
tech_report  = pd.read_csv('../data/technical_report.csv')

# Build lookup: symbol → sector + cap category (from tech_report)
fund_lookup = tech_report.set_index('Symbol')[['Sector', 'Cap Category', 'Market Cap Cr']].to_dict('index')

# Add fundamental score into lookup
for sym, row in fund_scores.set_index('Symbol').iterrows():
    if sym in fund_lookup:
        fund_lookup[sym]['Final Score'] = row['Final Score']

print(f"✅ Price data     : {len(price_data)} stocks")
print(f"✅ Indicator data : {len(indicator_data)} stocks")
print(f"✅ Fund scores    : {len(fund_scores)} stocks")
print(f"✅ Tech report    : {len(tech_report)} stocks")
print(f"✅ Lookup built   : {len(fund_lookup)} stocks")

print(f"\nSectors in universe:")
print(tech_report['Sector'].value_counts().to_string())

print(f"\nCap Categories:")
print(tech_report['Cap Category'].value_counts().to_string())

✅ Price data     : 37 stocks
✅ Indicator data : 37 stocks
✅ Fund scores    : 37 stocks
✅ Tech report    : 37 stocks
✅ Lookup built   : 37 stocks

Sectors in universe:
Sector
Information Technology            7
Financial Services                6
Chemicals                         6
Healthcare                        5
Consumer Durables                 4
Capital Goods                     4
Textiles                          2
Oil, Gas & Consumable Fuels       1
Consumer Services                 1
Automobile and Auto Components    1

Cap Categories:
Cap Category
Mid Cap           14
Large Cap         12
Mini Large Cap     7
Small Cap          4


In [13]:
# ── CELL 3: GROUP ASSIGNMENT ─────────────────────────────────

SECTOR_GROUP_MAP = {
    'Information Technology'         : 'IT',
    'Financial Services'             : 'Financial',
    'Chemicals'                      : 'Chemicals',
    'Healthcare'                     : 'Healthcare',
    'Consumer Durables'              : 'Consumer',
    'Consumer Services'              : 'Consumer',
    'Capital Goods'                  : 'Industrial',
    'Textiles'                       : 'Industrial',
    'Automobile and Auto Components' : 'Industrial',
    'Oil, Gas & Consumable Fuels'    : 'Industrial',
}

# Assign group to each symbol
symbol_group = {}
for sym, info in fund_lookup.items():
    sector = info['Sector']
    symbol_group[sym] = SECTOR_GROUP_MAP.get(sector, 'Industrial')

# Group → list of symbols
group_stocks = defaultdict(list)
for sym, grp in symbol_group.items():
    group_stocks[grp].append(sym)

print("Group assignments:")
for grp, stocks in sorted(group_stocks.items()):
    print(f"  {grp:12} ({len(stocks)} stocks): {', '.join(stocks)}")

Group assignments:
  Chemicals    (6 stocks): FINEORG, DHANUKA, PIIND, VINATIORGA, GALAXYSURF, RALLIS
  Consumer     (5 stocks): TITAN, HAVELLS, WONDERLA, VGUARD, SYMPHONY
  Financial    (6 stocks): CHOLAFIN, MUTHOOTFIN, ABSLAMC, HDFCBANK, CREDITACC, MANAPPURAM
  Healthcare   (5 stocks): SUNPHARMA, DIVISLAB, GRANULES, IPCALAB, APLLTD
  IT           (7 stocks): PERSISTENT, COFORGE, HAPPSTMNDS, TCS, INFY, LTTS, WIPRO
  Industrial   (8 stocks): BEL, GARFIBRES, PARAS, HAL, RELIANCE, SUPRAJIT, HIMATSEIDE, MIDHANI


In [15]:
# ── CELL 4: FEATURE ENGINEERING ──────────────────────────────
def build_features(symbol, price_df, ind_df):
    df = price_df.copy()
    
    # Drop overlapping columns from indicator df before joining
    overlap_cols = ['Open', 'High', 'Low', 'Close', 'Volume', 'Dividends', 'Stock Splits']
    ind_clean = ind_df.drop(columns=[c for c in overlap_cols if c in ind_df.columns])
    df = df.join(ind_clean, how='left')
    
    # ── PRICE FEATURES ───────────────────────────────────────
    df['return_1d']  = df['Close'].pct_change(1)
    df['return_5d']  = df['Close'].pct_change(5)
    df['return_20d'] = df['Close'].pct_change(20)
    df['return_60d'] = df['Close'].pct_change(60)

    df['high_52w'] = df['High'].rolling(252).max()
    df['low_52w']  = df['Low'].rolling(252).min()
    df['dist_52w_high'] = (df['Close'] - df['high_52w']) / df['high_52w']
    df['dist_52w_low']  = (df['Close'] - df['low_52w'])  / df['low_52w']

    df['tr'] = np.maximum(
        df['High'] - df['Low'],
        np.maximum(
            abs(df['High'] - df['Close'].shift(1)),
            abs(df['Low']  - df['Close'].shift(1))
        )
    )
    df['atr_14']         = df['tr'].rolling(14).mean()
    df['atr_pct']        = df['atr_14'] / df['Close']
    df['volatility_20d'] = df['return_1d'].rolling(20).std()
    df['volatility_60d'] = df['return_1d'].rolling(60).std()

    # ── VOLUME FEATURES ──────────────────────────────────────
    df['vol_ma20']      = df['Volume'].rolling(20).mean()
    df['vol_ratio_5d']  = df['Volume'].rolling(5).mean()  / df['vol_ma20']
    df['vol_ratio_20d'] = df['Volume'].rolling(20).mean() / df['Volume'].rolling(60).mean()
    df['obv']           = (np.sign(df['Close'].diff()) * df['Volume']).fillna(0).cumsum()
    df['obv_slope_10d'] = df['obv'].diff(10) / (df['obv'].abs().rolling(10).mean() + 1e-9)
    df['vol_spike']     = (df['Volume'] > df['vol_ma20'] * 2).astype(int)

    # ── RSI FEATURES ─────────────────────────────────────────
    if 'RSI' in df.columns:
        df['rsi']            = df['RSI']
        df['rsi_slope_5d']   = df['RSI'].diff(5)
        df['rsi_oversold']   = (df['RSI'] < 30).astype(int)
        df['rsi_overbought'] = (df['RSI'] > 70).astype(int)

    # ── MACD FEATURES ────────────────────────────────────────
    if 'MACD_Hist' in df.columns:
        df['macd_hist']     = df['MACD_Hist']
        df['macd_slope_3d'] = df['MACD_Hist'].diff(3)
        df['macd_slope_5d'] = df['MACD_Hist'].diff(5)
        df['macd_cross']    = (
            (df['MACD_Hist'] > 0) & (df['MACD_Hist'].shift(1) <= 0)
        ).astype(int)

    # ── ADX FEATURES ─────────────────────────────────────────
    if 'ADX' in df.columns:
        df['adx']       = df['ADX']
        df['adx_slope'] = df['ADX'].diff(5)
        df['di_spread'] = df['DI+'] - df['DI-'] if 'DI+' in df.columns else 0

    # ── EMA FEATURES ─────────────────────────────────────────
    if 'EMA20' in df.columns:
        df['price_vs_ema20']  = (df['Close'] - df['EMA20'])  / df['EMA20']
        df['price_vs_ema50']  = (df['Close'] - df['EMA50'])  / df['EMA50']
        df['price_vs_ema200'] = (df['Close'] - df['EMA200']) / df['EMA200']
        df['ema20_vs_ema50']  = (df['EMA20'] - df['EMA50'])  / df['EMA50']
        df['ema50_vs_ema200'] = (df['EMA50'] - df['EMA200']) / df['EMA200']

    # ── TARGET VARIABLES ─────────────────────────────────────
    df['future_return_20d'] = df['Close'].shift(-20) / df['Close'] - 1
    df['future_return_5d']  = df['Close'].shift(-5)  / df['Close'] - 1

    df['bottom_reversal'] = (
        (df['return_20d'] < -0.05) &
        (df['future_return_20d'] > 0.08)
    ).astype(int)

    df['top_reversal'] = (
        (df['return_20d'] > 0.05) &
        (df['future_return_20d'] < -0.08)
    ).astype(int)

    def assign_trend(row):
        r20          = row.get('return_20d', 0)
        ema20_vs_50  = row.get('ema20_vs_ema50', 0)
        ema50_vs_200 = row.get('ema50_vs_ema200', 0)
        if r20 > 0.08 and ema20_vs_50 > 0 and ema50_vs_200 > 0:
            return 'Strong Uptrend'
        elif r20 > 0.02 and ema20_vs_50 > 0:
            return 'Weak Uptrend'
        elif r20 < -0.08 and ema20_vs_50 < 0 and ema50_vs_200 < 0:
            return 'Strong Downtrend'
        elif r20 < -0.02 and ema20_vs_50 < 0:
            return 'Weak Downtrend'
        else:
            return 'Sideways'

    df['trend_label'] = df.apply(assign_trend, axis=1)

    df['target_25d']  = df['Close'].shift(-25) / df['Close'] - 1
    df['target_45d']  = df['Close'].shift(-45) / df['Close'] - 1
    df['target_180d'] = df['Close'].shift(-180) / df['Close'] - 1

    return df

# ── BUILD FEATURES FOR ALL STOCKS ────────────────────────────
print("Building features for all stocks...")
all_features = {}

for symbol in price_data.keys():
    price_df = price_data[symbol]
    ind_df   = indicator_data.get(symbol, pd.DataFrame())
    feat_df  = build_features(symbol, price_df, ind_df)
    all_features[symbol] = feat_df

print(f"✅ Features built for {len(all_features)} stocks")

sample    = list(all_features.keys())[0]
sample_df = all_features[sample].dropna(subset=['return_20d'])
print(f"\nSample stock  : {sample}")
print(f"Rows (all)    : {len(all_features[sample])}")
print(f"Rows (usable) : {len(sample_df)}")
print(f"Columns total : {len(sample_df.columns)}")

# Check what indicator columns came through
ind_cols = [c for c in sample_df.columns if c not in 
            ['Open','High','Low','Close','Volume','Dividends','Stock Splits']]
print(f"\nFeature columns ({len(ind_cols)}):")
print(ind_cols)

Building features for all stocks...
✅ Features built for 37 stocks

Sample stock  : PERSISTENT
Rows (all)    : 3943
Rows (usable) : 3923
Columns total : 62

Feature columns (55):
['EMA20', 'EMA50', 'EMA200', 'RSI', 'MACD', 'Signal', 'MACD_Hist', 'ADX', 'DI_Plus', 'DI_Minus', 'Vol_MA20', 'Vol_Ratio', 'return_1d', 'return_5d', 'return_20d', 'return_60d', 'high_52w', 'low_52w', 'dist_52w_high', 'dist_52w_low', 'tr', 'atr_14', 'atr_pct', 'volatility_20d', 'volatility_60d', 'vol_ma20', 'vol_ratio_5d', 'vol_ratio_20d', 'obv', 'obv_slope_10d', 'vol_spike', 'rsi', 'rsi_slope_5d', 'rsi_oversold', 'rsi_overbought', 'macd_hist', 'macd_slope_3d', 'macd_slope_5d', 'macd_cross', 'adx', 'adx_slope', 'di_spread', 'price_vs_ema20', 'price_vs_ema50', 'price_vs_ema200', 'ema20_vs_ema50', 'ema50_vs_ema200', 'future_return_20d', 'future_return_5d', 'bottom_reversal', 'top_reversal', 'trend_label', 'target_25d', 'target_45d', 'target_180d']


In [16]:
# ── CELL 5: DEFINE FEATURE COLUMNS ───────────────────────────

# Fix di_spread for all stocks (was using wrong column name)
for symbol in all_features:
    df = all_features[symbol]
    if 'DI_Plus' in df.columns and 'DI_Minus' in df.columns:
        df['di_spread'] = df['DI_Plus'] - df['DI_Minus']
    all_features[symbol] = df

# These are the input features for all 3 models
FEATURE_COLS = [
    # Price
    'return_1d', 'return_5d', 'return_20d', 'return_60d',
    'dist_52w_high', 'dist_52w_low', 'atr_pct',
    'volatility_20d', 'volatility_60d',
    # Volume
    'vol_ratio_5d', 'vol_ratio_20d', 'obv_slope_10d', 'vol_spike',
    # RSI
    'rsi', 'rsi_slope_5d', 'rsi_oversold', 'rsi_overbought',
    # MACD
    'macd_hist', 'macd_slope_3d', 'macd_slope_5d', 'macd_cross',
    # ADX
    'adx', 'adx_slope', 'di_spread',
    # EMA
    'price_vs_ema20', 'price_vs_ema50', 'price_vs_ema200',
    'ema20_vs_ema50', 'ema50_vs_ema200',
]

# Target columns
TARGET_BOTTOM   = 'bottom_reversal'   # Model 1a
TARGET_TOP      = 'top_reversal'      # Model 1b
TARGET_TREND    = 'trend_label'       # Model 2
TARGET_25D      = 'target_25d'        # Model 3a
TARGET_45D      = 'target_45d'        # Model 3b
TARGET_180D     = 'target_180d'       # Model 3c

print(f"✅ Feature columns defined: {len(FEATURE_COLS)}")
print(f"\nFeatures:")
for i, f in enumerate(FEATURE_COLS, 1):
    print(f"  {i:2}. {f}")

✅ Feature columns defined: 29

Features:
   1. return_1d
   2. return_5d
   3. return_20d
   4. return_60d
   5. dist_52w_high
   6. dist_52w_low
   7. atr_pct
   8. volatility_20d
   9. volatility_60d
  10. vol_ratio_5d
  11. vol_ratio_20d
  12. obv_slope_10d
  13. vol_spike
  14. rsi
  15. rsi_slope_5d
  16. rsi_oversold
  17. rsi_overbought
  18. macd_hist
  19. macd_slope_3d
  20. macd_slope_5d
  21. macd_cross
  22. adx
  23. adx_slope
  24. di_spread
  25. price_vs_ema20
  26. price_vs_ema50
  27. price_vs_ema200
  28. ema20_vs_ema50
  29. ema50_vs_ema200


In [17]:
# ── CELL 6: WALK-FORWARD TRAINING HELPER ─────────────────────

# Walk-forward folds: train up to year X, test on year X+1
WF_FOLDS = [
    ('2015-01-01', '2020-12-31', '2021-01-01', '2021-12-31'),
    ('2015-01-01', '2021-12-31', '2022-01-01', '2022-12-31'),
    ('2015-01-01', '2022-12-31', '2023-01-01', '2023-12-31'),
    ('2015-01-01', '2023-12-31', '2024-01-01', '2024-12-31'),
    ('2015-01-01', '2024-12-31', '2025-01-01', '2025-12-31'),
]

def prepare_group_data(group_name, target_col, min_rows=200):
    """
    Combine all stocks in a group into one training DataFrame.
    Adds symbol as a feature via label encoding.
    Drops rows with NaN in features or target.
    """
    group_syms = group_stocks[group_name]
    frames = []
    
    for sym in group_syms:
        df = all_features[sym].copy()
        df['symbol_id'] = sym
        # Keep only needed columns
        keep = FEATURE_COLS + [target_col, 'symbol_id']
        df = df[keep].copy()
        df = df.replace([np.inf, -np.inf], np.nan)
        df = df.dropna()
        df.index = all_features[sym].dropna(subset=FEATURE_COLS + [target_col]).index
        frames.append(df)
    
    if not frames:
        return None
    
    combined = pd.concat(frames)
    combined = combined.sort_index()
    
    # Encode symbol as integer feature
    le = LabelEncoder()
    combined['symbol_enc'] = le.fit_transform(combined['symbol_id'])
    
    return combined, le

def walk_forward_classify(group_name, target_col, model_params=None):
    """
    Walk-forward validation for classifier.
    Returns: trained model (on full data), metrics per fold
    """
    if model_params is None:
        model_params = {
            'n_estimators'    : 200,
            'max_depth'       : 4,
            'learning_rate'   : 0.05,
            'subsample'       : 0.8,
            'colsample_bytree': 0.8,
            'use_label_encoder': False,
            'eval_metric'     : 'logloss',
            'random_state'    : 42,
        }
    
    result = prepare_group_data(group_name, target_col)
    if result is None:
        print(f"  ⚠️  {group_name}: no data")
        return None, []
    
    combined, le = result
    feat_cols = FEATURE_COLS + ['symbol_enc']
    
    fold_metrics = []
    
    for fold_i, (tr_start, tr_end, te_start, te_end) in enumerate(WF_FOLDS):
        train_df = combined[tr_start:tr_end]
        test_df  = combined[te_start:te_end]
        
        if len(train_df) < 100 or len(test_df) < 10:
            continue
        
        X_train = train_df[feat_cols]
        y_train = train_df[target_col]
        X_test  = test_df[feat_cols]
        y_test  = test_df[target_col]
        
        # Skip fold if only one class in train
        if y_train.nunique() < 2:
            continue
        
        model = XGBClassifier(**model_params)
        model.fit(X_train, y_train,
                  eval_set=[(X_test, y_test)],
                  verbose=False)
        
        preds = model.predict(X_test)
        acc   = accuracy_score(y_test, preds)
        
        # Positive class metrics (reversal events are rare)
        pos_count = y_test.sum()
        fold_metrics.append({
            'fold'     : f"{te_start[:4]}",
            'accuracy' : round(acc, 3),
            'test_rows': len(y_test),
            'pos_events': int(pos_count),
        })
    
    # Final model: train on ALL data
    X_all = combined[feat_cols]
    y_all = combined[target_col]
    
    final_model = XGBClassifier(**model_params)
    final_model.fit(X_all, y_all, verbose=False)
    
    return final_model, fold_metrics, le

def walk_forward_regress(group_name, target_col, model_params=None):
    """
    Walk-forward validation for regressor.
    Returns: trained model, metrics per fold
    """
    if model_params is None:
        model_params = {
            'n_estimators'    : 200,
            'max_depth'       : 4,
            'learning_rate'   : 0.05,
            'subsample'       : 0.8,
            'colsample_bytree': 0.8,
            'random_state'    : 42,
        }
    
    result = prepare_group_data(group_name, target_col)
    if result is None:
        return None, []
    
    combined, le = result
    feat_cols = FEATURE_COLS + ['symbol_enc']
    
    fold_metrics = []
    
    for fold_i, (tr_start, tr_end, te_start, te_end) in enumerate(WF_FOLDS):
        train_df = combined[tr_start:tr_end]
        test_df  = combined[te_start:te_end]
        
        if len(train_df) < 100 or len(test_df) < 10:
            continue
        
        X_train = train_df[feat_cols]
        y_train = train_df[target_col]
        X_test  = test_df[feat_cols]
        y_test  = test_df[target_col]
        
        model = XGBRegressor(**model_params)
        model.fit(X_train, y_train,
                  eval_set=[(X_test, y_test)],
                  verbose=False)
        
        preds     = model.predict(X_test)
        # Directional accuracy: did we get the direction right?
        dir_acc   = np.mean(np.sign(preds) == np.sign(y_test))
        mae       = np.mean(np.abs(preds - y_test))
        
        fold_metrics.append({
            'fold'    : f"{te_start[:4]}",
            'dir_acc' : round(dir_acc, 3),
            'mae'     : round(mae, 3),
            'test_rows': len(y_test),
        })
    
    # Final model on all data
    X_all = combined[feat_cols]
    y_all = combined[target_col]
    
    final_model = XGBRegressor(**model_params)
    final_model.fit(X_all, y_all, verbose=False)
    
    return final_model, fold_metrics, le

print("✅ Walk-forward helpers defined")
print(f"   Folds: {len(WF_FOLDS)} (2021 → 2025)")
print(f"   Groups: {list(group_stocks.keys())}")

✅ Walk-forward helpers defined
   Folds: 5 (2021 → 2025)
   Groups: ['IT', 'Financial', 'Chemicals', 'Consumer', 'Industrial', 'Healthcare']


In [18]:
# ── CELL 7: TRAIN MODEL 1 — REVERSAL CLASSIFIER ──────────────
print("=" * 60)
print("MODEL 1 — REVERSAL CLASSIFIER")
print("=" * 60)

bottom_models = {}   # group → model
top_models    = {}   # group → model
bottom_encoders = {}
top_encoders    = {}
all_reversal_metrics = []

for group in group_stocks.keys():
    stocks = group_stocks[group]
    print(f"\n── {group} ({len(stocks)} stocks) ──")
    
    # Bottom reversal
    model_b, metrics_b, le_b = walk_forward_classify(group, TARGET_BOTTOM)
    if model_b:
        bottom_models[group]   = model_b
        bottom_encoders[group] = le_b
        print(f"  Bottom reversal — walk-forward:")
        for m in metrics_b:
            print(f"    {m['fold']}: acc={m['accuracy']} | test={m['test_rows']} rows | events={m['pos_events']}")
    
    # Top reversal
    model_t, metrics_t, le_t = walk_forward_classify(group, TARGET_TOP)
    if model_t:
        top_models[group]   = model_t
        top_encoders[group] = le_t
        print(f"  Top reversal — walk-forward:")
        for m in metrics_t:
            print(f"    {m['fold']}: acc={m['accuracy']} | test={m['test_rows']} rows | events={m['pos_events']}")
    
    # Store metrics
    for m in metrics_b:
        all_reversal_metrics.append({**m, 'group': group, 'model': 'bottom_reversal'})
    for m in metrics_t:
        all_reversal_metrics.append({**m, 'group': group, 'model': 'top_reversal'})

print(f"\n✅ Model 1 trained for {len(bottom_models)} groups")

MODEL 1 — REVERSAL CLASSIFIER

── IT (7 stocks) ──
  Bottom reversal — walk-forward:
    2021: acc=0.978 | test=1558 rows | events=27
    2022: acc=0.888 | test=1736 rows | events=112
    2023: acc=0.96 | test=1715 rows | events=70
    2024: acc=0.969 | test=1722 rows | events=48
    2025: acc=0.95 | test=1743 rows | events=73
  Top reversal — walk-forward:
    2021: acc=0.922 | test=1558 rows | events=44
    2022: acc=0.909 | test=1736 rows | events=158
    2023: acc=0.984 | test=1715 rows | events=30
    2024: acc=0.952 | test=1722 rows | events=66
    2025: acc=0.947 | test=1743 rows | events=73

── Financial (6 stocks) ──
  Bottom reversal — walk-forward:
    2021: acc=0.926 | test=1240 rows | events=68
    2022: acc=0.889 | test=1294 rows | events=50
    2023: acc=0.938 | test=1470 rows | events=86
    2024: acc=0.938 | test=1476 rows | events=86
    2025: acc=0.959 | test=1494 rows | events=34
  Top reversal — walk-forward:
    2021: acc=0.926 | test=1240 rows | events=89
    202

In [19]:
# ── CELL 8: TRAIN MODEL 2 — TREND CLASSIFIER ─────────────────
print("=" * 60)
print("MODEL 2 — TREND CLASSIFIER (5-class)")
print("=" * 60)

trend_model_params = {
    'n_estimators'    : 300,
    'max_depth'       : 5,
    'learning_rate'   : 0.05,
    'subsample'       : 0.8,
    'colsample_bytree': 0.8,
    'use_label_encoder': False,
    'eval_metric'     : 'mlogloss',
    'random_state'    : 42,
}

trend_models   = {}
trend_encoders = {}
trend_label_encoders = {}
all_trend_metrics = []

for group in group_stocks.keys():
    stocks = group_stocks[group]
    print(f"\n── {group} ({len(stocks)} stocks) ──")
    
    result = prepare_group_data(group, TARGET_TREND)
    if result is None:
        continue
    
    combined, le_sym = result
    feat_cols = FEATURE_COLS + ['symbol_enc']
    
    # Encode trend labels to integers
    le_trend = LabelEncoder()
    combined['trend_enc'] = le_trend.fit_transform(combined[TARGET_TREND])
    
    fold_metrics = []
    
    for tr_start, tr_end, te_start, te_end in WF_FOLDS:
        train_df = combined[tr_start:tr_end]
        test_df  = combined[te_start:te_end]
        
        if len(train_df) < 100 or len(test_df) < 10:
            continue
        if train_df['trend_enc'].nunique() < 2:
            continue
        
        X_train = train_df[feat_cols]
        y_train = train_df['trend_enc']
        X_test  = test_df[feat_cols]
        y_test  = test_df['trend_enc']
        
        model = XGBClassifier(**trend_model_params)
        model.fit(X_train, y_train,
                  eval_set=[(X_test, y_test)],
                  verbose=False)
        
        preds = model.predict(X_test)
        acc   = accuracy_score(y_test, preds)
        
        fold_metrics.append({
            'fold'     : te_start[:4],
            'accuracy' : round(acc, 3),
            'test_rows': len(y_test),
        })
        print(f"  {te_start[:4]}: acc={round(acc,3)} | test={len(y_test)} rows")
    
    # Train final model on all data
    X_all = combined[feat_cols]
    y_all = combined['trend_enc']
    
    final_model = XGBClassifier(**trend_model_params)
    final_model.fit(X_all, y_all, verbose=False)
    
    trend_models[group]        = final_model
    trend_encoders[group]      = le_sym
    trend_label_encoders[group]= le_trend
    
    for m in fold_metrics:
        all_trend_metrics.append({**m, 'group': group})

print(f"\n✅ Model 2 trained for {len(trend_models)} groups")

MODEL 2 — TREND CLASSIFIER (5-class)

── IT (7 stocks) ──
  2021: acc=0.999 | test=1558 rows
  2022: acc=0.994 | test=1736 rows
  2023: acc=0.998 | test=1715 rows
  2024: acc=0.999 | test=1722 rows
  2025: acc=0.998 | test=1743 rows

── Financial (6 stocks) ──
  2021: acc=0.997 | test=1240 rows
  2022: acc=0.998 | test=1294 rows
  2023: acc=0.995 | test=1470 rows
  2024: acc=0.995 | test=1476 rows
  2025: acc=0.996 | test=1494 rows

── Chemicals (6 stocks) ──
  2021: acc=0.997 | test=1488 rows
  2022: acc=0.999 | test=1488 rows
  2023: acc=0.996 | test=1470 rows
  2024: acc=0.999 | test=1476 rows
  2025: acc=0.995 | test=1494 rows

── Consumer (5 stocks) ──
  2021: acc=0.998 | test=1240 rows
  2022: acc=0.993 | test=1240 rows
  2023: acc=0.999 | test=1225 rows
  2024: acc=0.998 | test=1230 rows
  2025: acc=0.998 | test=1245 rows

── Industrial (8 stocks) ──
  2021: acc=0.999 | test=1736 rows
  2022: acc=0.997 | test=1796 rows
  2023: acc=0.998 | test=1960 rows
  2024: acc=0.995 | test=

In [20]:
# ── CELL 9: TRAIN MODEL 3 — PRICE FORECAST (REGRESSOR) ───────
print("=" * 60)
print("MODEL 3 — PRICE FORECAST REGRESSOR")
print("=" * 60)

forecast_models    = {}   # group → {25d, 45d, 180d}
forecast_encoders  = {}
all_forecast_metrics = []

for group in group_stocks.keys():
    stocks = group_stocks[group]
    print(f"\n── {group} ({len(stocks)} stocks) ──")
    
    forecast_models[group]   = {}
    forecast_encoders[group] = {}
    
    for horizon, target_col in [('25d', TARGET_25D), ('45d', TARGET_45D), ('180d', TARGET_180D)]:
        model_f, metrics_f, le_f = walk_forward_regress(group, target_col)
        
        if model_f:
            forecast_models[group][horizon]   = model_f
            forecast_encoders[group][horizon] = le_f
            
            dir_accs = [m['dir_acc'] for m in metrics_f]
            avg_dir  = round(np.mean(dir_accs), 3) if dir_accs else 0
            print(f"  {horizon}: avg dir_acc={avg_dir} | folds={len(metrics_f)}")
            
            for m in metrics_f:
                all_forecast_metrics.append({
                    **m, 'group': group, 'horizon': horizon
                })

print(f"\n✅ Model 3 trained for {len(forecast_models)} groups")

MODEL 3 — PRICE FORECAST REGRESSOR

── IT (7 stocks) ──
  25d: avg dir_acc=0.492 | folds=5
  45d: avg dir_acc=0.511 | folds=5
  180d: avg dir_acc=0.538 | folds=5

── Financial (6 stocks) ──
  25d: avg dir_acc=0.548 | folds=5
  45d: avg dir_acc=0.559 | folds=5
  180d: avg dir_acc=0.703 | folds=5

── Chemicals (6 stocks) ──
  25d: avg dir_acc=0.505 | folds=5
  45d: avg dir_acc=0.533 | folds=5
  180d: avg dir_acc=0.48 | folds=5

── Consumer (5 stocks) ──
  25d: avg dir_acc=0.495 | folds=5
  45d: avg dir_acc=0.511 | folds=5
  180d: avg dir_acc=0.561 | folds=5

── Industrial (8 stocks) ──
  25d: avg dir_acc=0.519 | folds=5
  45d: avg dir_acc=0.531 | folds=5
  180d: avg dir_acc=0.636 | folds=5

── Healthcare (5 stocks) ──
  25d: avg dir_acc=0.546 | folds=5
  45d: avg dir_acc=0.514 | folds=5
  180d: avg dir_acc=0.556 | folds=5

✅ Model 3 trained for 6 groups


In [21]:
# ── CELL 10: FIX TREND LABELS (NO LEAKAGE) ───────────────────
# Old definition used current indicators — leaked into features
# New definition uses FUTURE price movement — true forward label

def assign_trend_forward(row):
    """
    Trend label based on what actually happens in next 20 days.
    No overlap with any feature column.
    """
    f20 = row.get('future_return_20d', 0)  # already calculated, shift(-20)
    f5  = row.get('future_return_5d', 0)
    
    if pd.isna(f20):
        return 'Sideways'
    if f20 > 0.10:
        return 'Strong Uptrend'
    elif f20 > 0.03:
        return 'Weak Uptrend'
    elif f20 < -0.10:
        return 'Strong Downtrend'
    elif f20 < -0.03:
        return 'Weak Downtrend'
    else:
        return 'Sideways'

# Rebuild trend labels for all stocks
for symbol in all_features:
    df = all_features[symbol]
    df['trend_label'] = df.apply(assign_trend_forward, axis=1)
    all_features[symbol] = df

# Verify distribution on one stock
sample = list(all_features.keys())[0]
print(f"Trend label distribution ({sample}):")
print(all_features[sample]['trend_label'].value_counts())

Trend label distribution (PERSISTENT):
trend_label
Sideways            1045
Weak Uptrend         953
Strong Uptrend       853
Weak Downtrend       759
Strong Downtrend     333
Name: count, dtype: int64


In [22]:
# ── CELL 11: REBUILD MODEL 1 WITH PROPER METRICS + BALANCING ─
from sklearn.metrics import precision_score, recall_score, f1_score

def walk_forward_classify_v2(group_name, target_col):
    """
    Improved reversal classifier:
    - scale_pos_weight to handle class imbalance
    - Reports precision + recall on positive class (not just accuracy)
    """
    result = prepare_group_data(group_name, target_col)
    if result is None:
        return None, [], None
    
    combined, le = result
    feat_cols = FEATURE_COLS + ['symbol_enc']
    fold_metrics = []
    
    for tr_start, tr_end, te_start, te_end in WF_FOLDS:
        train_df = combined[tr_start:tr_end]
        test_df  = combined[te_start:te_end]
        
        if len(train_df) < 100 or len(test_df) < 10:
            continue
        if train_df[target_col].nunique() < 2:
            continue
        
        X_train = train_df[feat_cols]
        y_train = train_df[target_col]
        X_test  = test_df[feat_cols]
        y_test  = test_df[target_col]
        
        # Handle class imbalance — weight positive class higher
        neg_count = (y_train == 0).sum()
        pos_count = (y_train == 1).sum()
        scale_pos = neg_count / pos_count if pos_count > 0 else 1
        
        model = XGBClassifier(
            n_estimators    = 200,
            max_depth       = 4,
            learning_rate   = 0.05,
            subsample       = 0.8,
            colsample_bytree= 0.8,
            scale_pos_weight= scale_pos,   # KEY FIX
            use_label_encoder=False,
            eval_metric     = 'aucpr',     # area under precision-recall curve
            random_state    = 42,
        )
        model.fit(X_train, y_train,
                  eval_set=[(X_test, y_test)],
                  verbose=False)
        
        preds    = model.predict(X_test)
        probs    = model.predict_proba(X_test)[:, 1]
        
        acc  = accuracy_score(y_test, preds)
        prec = precision_score(y_test, preds, zero_division=0)
        rec  = recall_score(y_test, preds, zero_division=0)
        f1   = f1_score(y_test, preds, zero_division=0)
        
        pos_test = int(y_test.sum())
        fold_metrics.append({
            'fold'      : te_start[:4],
            'accuracy'  : round(acc, 3),
            'precision' : round(prec, 3),
            'recall'    : round(rec, 3),
            'f1'        : round(f1, 3),
            'test_rows' : len(y_test),
            'pos_events': pos_test,
        })
    
    # Final model on all data
    neg_all = (combined[target_col] == 0).sum()
    pos_all = (combined[target_col] == 1).sum()
    scale_all = neg_all / pos_all if pos_all > 0 else 1
    
    final_model = XGBClassifier(
        n_estimators    = 200,
        max_depth       = 4,
        learning_rate   = 0.05,
        subsample       = 0.8,
        colsample_bytree= 0.8,
        scale_pos_weight= scale_all,
        use_label_encoder=False,
        eval_metric     = 'aucpr',
        random_state    = 42,
    )
    final_model.fit(combined[feat_cols], combined[target_col], verbose=False)
    
    return final_model, fold_metrics, le

# ── RETRAIN MODEL 1 ───────────────────────────────────────────
print("=" * 60)
print("MODEL 1 REBUILD — REVERSAL CLASSIFIER (with class balance)")
print("=" * 60)

bottom_models   = {}
top_models      = {}
bottom_encoders = {}
top_encoders    = {}

for group in group_stocks.keys():
    print(f"\n── {group} ──")
    
    model_b, metrics_b, le_b = walk_forward_classify_v2(group, TARGET_BOTTOM)
    if model_b:
        bottom_models[group]   = model_b
        bottom_encoders[group] = le_b
        print(f"  Bottom reversal:")
        for m in metrics_b:
            print(f"    {m['fold']}: prec={m['precision']} | rec={m['recall']} | f1={m['f1']} | events={m['pos_events']}/{m['test_rows']}")
    
    model_t, metrics_t, le_t = walk_forward_classify_v2(group, TARGET_TOP)
    if model_t:
        top_models[group]   = model_t
        top_encoders[group] = le_t
        print(f"  Top reversal:")
        for m in metrics_t:
            print(f"    {m['fold']}: prec={m['precision']} | rec={m['recall']} | f1={m['f1']} | events={m['pos_events']}/{m['test_rows']}")

print(f"\n✅ Model 1 rebuilt for {len(bottom_models)} groups")

MODEL 1 REBUILD — REVERSAL CLASSIFIER (with class balance)

── IT ──
  Bottom reversal:
    2021: prec=0.239 | rec=0.407 | f1=0.301 | events=27/1558
    2022: prec=0.205 | rec=0.875 | f1=0.332 | events=112/1736
    2023: prec=0.261 | rec=0.529 | f1=0.349 | events=70/1715
    2024: prec=0.11 | rec=0.417 | f1=0.174 | events=48/1722
    2025: prec=0.178 | rec=0.849 | f1=0.295 | events=73/1743
  Top reversal:
    2021: prec=0.016 | rec=0.136 | f1=0.028 | events=44/1558
    2022: prec=0.582 | rec=0.335 | f1=0.426 | events=158/1736
    2023: prec=0.072 | rec=0.633 | f1=0.129 | events=30/1715
    2024: prec=0.106 | rec=0.53 | f1=0.176 | events=66/1722
    2025: prec=0.179 | rec=0.479 | f1=0.26 | events=73/1743

── Financial ──
  Bottom reversal:
    2021: prec=0.228 | rec=0.676 | f1=0.341 | events=68/1240
    2022: prec=0.132 | rec=0.6 | f1=0.216 | events=50/1294
    2023: prec=0.434 | rec=0.419 | f1=0.426 | events=86/1470
    2024: prec=0.255 | rec=0.767 | f1=0.383 | events=86/1476
    2025:

In [23]:
# ── CELL 12: RETRAIN MODEL 2 — FIXED TREND LABELS ────────────
print("=" * 60)
print("MODEL 2 REBUILD — TREND CLASSIFIER (no leakage)")
print("=" * 60)

trend_models         = {}
trend_encoders       = {}
trend_label_encoders = {}
all_trend_metrics    = []

for group in group_stocks.keys():
    print(f"\n── {group} ──")
    
    result = prepare_group_data(group, TARGET_TREND)
    if result is None:
        continue
    
    combined, le_sym = result
    feat_cols = FEATURE_COLS + ['symbol_enc']
    
    le_trend = LabelEncoder()
    combined['trend_enc'] = le_trend.fit_transform(combined[TARGET_TREND])
    
    fold_metrics = []
    
    for tr_start, tr_end, te_start, te_end in WF_FOLDS:
        train_df = combined[tr_start:tr_end]
        test_df  = combined[te_start:te_end]
        
        if len(train_df) < 100 or len(test_df) < 10:
            continue
        if train_df['trend_enc'].nunique() < 2:
            continue
        
        X_train = train_df[feat_cols]
        y_train = train_df['trend_enc']
        X_test  = test_df[feat_cols]
        y_test  = test_df['trend_enc']
        
        model = XGBClassifier(
            n_estimators    = 300,
            max_depth       = 5,
            learning_rate   = 0.05,
            subsample       = 0.8,
            colsample_bytree= 0.8,
            use_label_encoder=False,
            eval_metric     = 'mlogloss',
            random_state    = 42,
        )
        model.fit(X_train, y_train,
                  eval_set=[(X_test, y_test)],
                  verbose=False)
        
        preds = model.predict(X_test)
        acc   = accuracy_score(y_test, preds)
        
        # Per-class accuracy
        report = classification_report(
            y_test, preds,
            target_names=le_trend.classes_,
            output_dict=True,
            zero_division=0
        )
        
        fold_metrics.append({
            'fold'     : te_start[:4],
            'accuracy' : round(acc, 3),
            'test_rows': len(y_test),
        })
        print(f"  {te_start[:4]}: acc={round(acc,3)} | test={len(y_test)} rows")
    
    X_all = combined[feat_cols]
    y_all = combined['trend_enc']
    
    final_model = XGBClassifier(
        n_estimators    = 300,
        max_depth       = 5,
        learning_rate   = 0.05,
        subsample       = 0.8,
        colsample_bytree= 0.8,
        use_label_encoder=False,
        eval_metric     = 'mlogloss',
        random_state    = 42,
    )
    final_model.fit(X_all, y_all, verbose=False)
    
    trend_models[group]         = final_model
    trend_encoders[group]       = le_sym
    trend_label_encoders[group] = le_trend
    
    for m in fold_metrics:
        all_trend_metrics.append({**m, 'group': group})

print(f"\n✅ Model 2 rebuilt for {len(trend_models)} groups")

MODEL 2 REBUILD — TREND CLASSIFIER (no leakage)

── IT ──
  2021: acc=0.25 | test=1558 rows
  2022: acc=0.219 | test=1736 rows
  2023: acc=0.294 | test=1715 rows
  2024: acc=0.271 | test=1722 rows
  2025: acc=0.247 | test=1743 rows

── Financial ──
  2021: acc=0.237 | test=1240 rows
  2022: acc=0.26 | test=1294 rows
  2023: acc=0.276 | test=1470 rows
  2024: acc=0.287 | test=1476 rows
  2025: acc=0.325 | test=1494 rows

── Chemicals ──
  2021: acc=0.25 | test=1488 rows
  2022: acc=0.192 | test=1488 rows
  2023: acc=0.233 | test=1470 rows
  2024: acc=0.245 | test=1476 rows
  2025: acc=0.294 | test=1494 rows

── Consumer ──
  2021: acc=0.251 | test=1240 rows
  2022: acc=0.234 | test=1240 rows
  2023: acc=0.227 | test=1225 rows
  2024: acc=0.25 | test=1230 rows
  2025: acc=0.269 | test=1245 rows

── Industrial ──
  2021: acc=0.272 | test=1736 rows
  2022: acc=0.248 | test=1796 rows
  2023: acc=0.282 | test=1960 rows
  2024: acc=0.22 | test=1968 rows
  2025: acc=0.29 | test=1992 rows

── H

In [24]:
# ── CELL 13: SIMPLIFY TREND TO 3 CLASSES ─────────────────────
# 5-class was too fine-grained — model barely beats random
# 3-class is more learnable and still actionable

def assign_trend_3class(row):
    f20 = row.get('future_return_20d', 0)
    if pd.isna(f20):
        return 'Sideways'
    if f20 > 0.05:
        return 'Uptrend'
    elif f20 < -0.05:
        return 'Downtrend'
    else:
        return 'Sideways'

for symbol in all_features:
    df = all_features[symbol]
    df['trend_label'] = df.apply(assign_trend_3class, axis=1)
    all_features[symbol] = df

# Verify distribution
sample = list(all_features.keys())[0]
print(f"3-class trend distribution ({sample}):")
print(all_features[sample]['trend_label'].value_counts())
print(f"\nRandom baseline accuracy: {1/3:.1%}")

3-class trend distribution (PERSISTENT):
trend_label
Sideways     1631
Uptrend      1487
Downtrend     825
Name: count, dtype: int64

Random baseline accuracy: 33.3%


In [26]:
# ── CELL 14: RETRAIN MODEL 2 — 3-CLASS TREND ─────────────────

print("=" * 60)
print("MODEL 2 RETRAIN — 3-CLASS TREND CLASSIFIER")
print("=" * 60)

trend_models         = {}
trend_encoders       = {}
trend_label_encoders = {}

for group in group_stocks.keys():
    print(f"\n── {group} ──")

    result = prepare_group_data(group, TARGET_TREND)
    if result is None:
        continue

    combined, le_sym = result
    feat_cols = FEATURE_COLS + ['symbol_enc']

    le_trend = LabelEncoder()
    combined['trend_enc'] = le_trend.fit_transform(combined[TARGET_TREND])

    print(f"  Classes: {list(le_trend.classes_)}")
    print(f"  Distribution: {dict(combined[TARGET_TREND].value_counts())}")

    fold_metrics = []

    for tr_start, tr_end, te_start, te_end in WF_FOLDS:
        train_df = combined[tr_start:tr_end]
        test_df  = combined[te_start:te_end]

        if len(train_df) < 100 or len(test_df) < 10:
            continue
        if train_df['trend_enc'].nunique() < 2:
            continue

        X_train = train_df[feat_cols]
        y_train = train_df['trend_enc']
        X_test  = test_df[feat_cols]
        y_test  = test_df['trend_enc']

        model = XGBClassifier(
            n_estimators     = 300,
            max_depth        = 5,
            learning_rate    = 0.05,
            subsample        = 0.8,
            colsample_bytree = 0.8,
            use_label_encoder= False,
            eval_metric      = 'mlogloss',
            random_state     = 42,
        )
        model.fit(X_train, y_train,
                  eval_set=[(X_test, y_test)],
                  verbose=False)

        preds = model.predict(X_test)
        acc   = accuracy_score(y_test, preds)

        fold_metrics.append({
            'fold'     : te_start[:4],
            'accuracy' : round(acc, 3),
            'test_rows': len(y_test),
        })
        print(f"  {te_start[:4]}: acc={round(acc,3)} | test={len(y_test)} rows")

    # Train final model on all data
    final_model = XGBClassifier(
        n_estimators     = 300,
        max_depth        = 5,
        learning_rate    = 0.05,
        subsample        = 0.8,
        colsample_bytree = 0.8,
        use_label_encoder= False,
        eval_metric      = 'mlogloss',
        random_state     = 42,
    )
    final_model.fit(combined[feat_cols], combined['trend_enc'], verbose=False)

    trend_models[group]         = final_model
    trend_encoders[group]       = le_sym
    trend_label_encoders[group] = le_trend

print(f"\n✅ Model 2 retrained for {len(trend_models)} groups")
print(f"   Random baseline: 33.3%")
print(f"   Target: >45% to be useful")

MODEL 2 RETRAIN — 3-CLASS TREND CLASSIFIER

── IT ──
  Classes: ['Downtrend', 'Sideways', 'Uptrend']
  Distribution: {'Sideways': 14163, 'Uptrend': 11220, 'Downtrend': 6677}
  2021: acc=0.357 | test=1558 rows
  2022: acc=0.372 | test=1736 rows
  2023: acc=0.458 | test=1715 rows
  2024: acc=0.426 | test=1722 rows
  2025: acc=0.424 | test=1743 rows

── Financial ──
  Classes: ['Downtrend', 'Sideways', 'Uptrend']
  Distribution: {'Sideways': 9934, 'Uptrend': 7977, 'Downtrend': 4585}
  2021: acc=0.401 | test=1240 rows
  2022: acc=0.406 | test=1294 rows
  2023: acc=0.446 | test=1470 rows
  2024: acc=0.436 | test=1476 rows
  2025: acc=0.56 | test=1494 rows

── Chemicals ──
  Classes: ['Downtrend', 'Sideways', 'Uptrend']
  Distribution: {'Sideways': 8679, 'Uptrend': 6625, 'Downtrend': 4374}
  2021: acc=0.433 | test=1488 rows
  2022: acc=0.322 | test=1488 rows
  2023: acc=0.386 | test=1470 rows
  2024: acc=0.402 | test=1476 rows
  2025: acc=0.452 | test=1494 rows

── Consumer ──
  Classes: ['D

In [27]:
# ── CELL 15: SAVE ALL MODELS ──────────────────────────────────

print("Saving models...")

# Save bottom reversal models
with open('../models/bottom_models.pkl', 'wb') as f:
    pickle.dump(bottom_models, f)

with open('../models/bottom_encoders.pkl', 'wb') as f:
    pickle.dump(bottom_encoders, f)

# Save top reversal models
with open('../models/top_models.pkl', 'wb') as f:
    pickle.dump(top_models, f)

with open('../models/top_encoders.pkl', 'wb') as f:
    pickle.dump(top_encoders, f)

# Save trend models
with open('../models/trend_models.pkl', 'wb') as f:
    pickle.dump(trend_models, f)

with open('../models/trend_encoders.pkl', 'wb') as f:
    pickle.dump(trend_encoders, f)

with open('../models/trend_label_encoders.pkl', 'wb') as f:
    pickle.dump(trend_label_encoders, f)

# Save forecast models
with open('../models/forecast_models.pkl', 'wb') as f:
    pickle.dump(forecast_models, f)

with open('../models/forecast_encoders.pkl', 'wb') as f:
    pickle.dump(forecast_encoders, f)

# Save group mapping (needed at inference time)
with open('../models/symbol_group.pkl', 'wb') as f:
    pickle.dump(symbol_group, f)

with open('../models/group_stocks.pkl', 'wb') as f:
    pickle.dump(dict(group_stocks), f)

# Verify
import os
saved = os.listdir('../models/')
print(f"\n✅ Models saved to /models/")
for f in sorted(saved):
    size = os.path.getsize(f'../models/{f}') / 1024
    print(f"  {f:40} {size:.1f} KB")

Saving models...

✅ Models saved to /models/
  bottom_encoders.pkl                      0.9 KB
  bottom_models.pkl                        1661.3 KB
  forecast_encoders.pkl                    2.3 KB
  forecast_models.pkl                      6004.1 KB
  group_stocks.pkl                         0.5 KB
  symbol_group.pkl                         0.5 KB
  top_encoders.pkl                         0.9 KB
  top_models.pkl                           1666.2 KB
  trend_encoders.pkl                       0.9 KB
  trend_label_encoders.pkl                 0.7 KB
  trend_models.pkl                         13522.8 KB


In [42]:
# ── CELL 16: INFERENCE ENGINE ─────────────────────────────────

def get_latest_features(symbol):
    df = all_features[symbol].copy()
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=FEATURE_COLS)
    if len(df) == 0:
        return None
    return df.iloc[[-1]]

def get_symbol_enc(symbol, group, encoder):
    try:
        return encoder.transform([symbol])[0]
    except:
        return 0

def run_inference(symbol):
    result = {
        'Symbol'             : symbol,
        'Group'              : None,
        'Bottom_Rev_Prob'    : None,
        'Top_Rev_Prob'       : None,
        'Trend_Label'        : None,
        'Trend_Conf'         : None,
        'Forecast_25d_Pct'   : None,
        'Forecast_45d_Pct'   : None,
        'Forecast_180d_Pct'  : None,
        'Forecast_25d_Price' : None,
        'Forecast_45d_Price' : None,
        'Forecast_180d_Price': None,
        'Bottom_Rev_Flag'    : False,
        'Top_Rev_Flag'       : False,
        'Show_180d'          : False,
    }

    group = symbol_group.get(symbol)
    if group is None:
        return result
    result['Group'] = group

    latest = get_latest_features(symbol)
    if latest is None:
        return result

    feat_cols = FEATURE_COLS + ['symbol_enc']

    # ── MODEL 1: REVERSAL ─────────────────────────────────────
    if group in bottom_models:
        sym_enc = get_symbol_enc(symbol, group, bottom_encoders[group])
        latest['symbol_enc'] = sym_enc
        X = latest[feat_cols]

        bottom_prob = bottom_models[group].predict_proba(X)[0][1]
        result['Bottom_Rev_Prob'] = round(float(bottom_prob) * 100, 1)
        result['Bottom_Rev_Flag'] = bottom_prob >= 0.60

        top_prob = top_models[group].predict_proba(X)[0][1]
        result['Top_Rev_Prob'] = round(float(top_prob) * 100, 1)
        result['Top_Rev_Flag'] = top_prob >= 0.60

    # ── MODEL 2: TREND ────────────────────────────────────────
    if group in trend_models:
        sym_enc = get_symbol_enc(symbol, group, trend_encoders[group])
        latest['symbol_enc'] = sym_enc
        X = latest[feat_cols]

        trend_probs = trend_models[group].predict_proba(X)[0]
        trend_pred  = trend_models[group].predict(X)[0]
        trend_conf  = round(float(max(trend_probs)) * 100, 1)

        # Confidence gate — below 55% model is not reliable
        if trend_conf >= 55:
            trend_label = trend_label_encoders[group].inverse_transform([trend_pred])[0]
        else:
            trend_label = 'Uncertain'

        result['Trend_Label'] = trend_label
        result['Trend_Conf']  = trend_conf

        # 180d only shown if Uptrend + confidence > 60%
        result['Show_180d'] = (trend_label == 'Uptrend' and trend_conf >= 60)

    # ── MODEL 3: FORECAST ─────────────────────────────────────
    if group in forecast_models:
        sym_enc = get_symbol_enc(symbol, group, forecast_encoders[group]['25d'])
        latest['symbol_enc'] = sym_enc
        X = latest[feat_cols]

        current_price = all_features[symbol]['Close'].iloc[-1]

        for horizon in ['25d', '45d', '180d']:
            if horizon not in forecast_models[group]:
                continue
            pred_return = forecast_models[group][horizon].predict(X)[0]
            pred_price  = current_price * (1 + pred_return)
            result[f'Forecast_{horizon}_Pct']  = round(float(pred_return) * 100, 1)
            result[f'Forecast_{horizon}_Price'] = round(float(pred_price), 2)

    return result

# ── TEST ON GALAXYSURF ────────────────────────────────────────
test_result = run_inference('GALAXYSURF')
print(f"GALAXYSURF inference:")
print(f"  Trend  : {test_result['Trend_Label']} ({test_result['Trend_Conf']}% conf)")
print(f"  Bottom : {test_result['Bottom_Rev_Prob']}%")
print(f"  Top    : {test_result['Top_Rev_Prob']}%")

GALAXYSURF inference:
  Trend  : Uncertain (49.7% conf)
  Bottom : 78.4%
  Top    : 0.0%


In [43]:
# ── CELL 17: RUN INFERENCE ON ALL 37 STOCKS ───────────────────

print("Running inference on all stocks...")
print("=" * 60)

all_ml_scores = []

for symbol in price_data.keys():
    result = run_inference(symbol)
    all_ml_scores.append(result)

ml_scores_df = pd.DataFrame(all_ml_scores)

# Save to CSV
ml_scores_df.to_csv('../data/ml_scores.csv', index=False)

print(f"✅ Inference complete for {len(ml_scores_df)} stocks")
print(f"\nML Scores Summary:")
print(f"  Stocks with Bottom Rev ⚡ : {ml_scores_df['Bottom_Rev_Flag'].sum()}")
print(f"  Stocks with Top Rev ⚡    : {ml_scores_df['Top_Rev_Flag'].sum()}")
print(f"  Stocks showing 180d forecast: {ml_scores_df['Show_180d'].sum()}")
print(f"\nTrend Distribution:")
print(ml_scores_df['Trend_Label'].value_counts().to_string())
print(f"\nTop Bottom Reversal candidates:")
top_bottom = ml_scores_df.nlargest(5, 'Bottom_Rev_Prob')[
    ['Symbol', 'Group', 'Trend_Label', 'Trend_Conf', 'Bottom_Rev_Prob', 'Forecast_25d_Pct']
]
print(top_bottom.to_string(index=False))
print(f"\nTop Top-Reversal candidates:")
top_top = ml_scores_df.nlargest(5, 'Top_Rev_Prob')[
    ['Symbol', 'Group', 'Trend_Label', 'Trend_Conf', 'Top_Rev_Prob', 'Forecast_25d_Pct']
]
print(top_top.to_string(index=False))

Running inference on all stocks...
✅ Inference complete for 37 stocks

ML Scores Summary:
  Stocks with Bottom Rev ⚡ : 16
  Stocks with Top Rev ⚡    : 0
  Stocks showing 180d forecast: 0

Trend Distribution:
Trend_Label
Sideways     22
Uncertain    15

Top Bottom Reversal candidates:
    Symbol      Group Trend_Label  Trend_Conf  Bottom_Rev_Prob  Forecast_25d_Pct
HIMATSEIDE Industrial   Uncertain        38.3             86.1               3.8
    RALLIS  Chemicals   Uncertain        48.5             84.7               4.1
  SYMPHONY   Consumer   Uncertain        50.1             84.2               4.1
   COFORGE         IT    Sideways        69.1             81.3               6.3
      INFY         IT    Sideways        65.0             79.3               4.0

Top Top-Reversal candidates:
    Symbol      Group Trend_Label  Trend_Conf  Top_Rev_Prob  Forecast_25d_Pct
  WONDERLA   Consumer    Sideways        73.5          58.2              -1.5
   IPCALAB Healthcare   Uncertain        54

In [46]:
# ── CELL 18 (UPDATED): GENERATE ML REPORT ────────────────────

from datetime import date

def generate_ml_report(ml_df, tech_df):
    today = date.today().strftime("%d %B %Y")
    merged = tech_df.merge(ml_df, on='Symbol', how='left')

    lines = []
    lines.append("═" * 78)
    lines.append(f"  ML SIGNALS — {today}")
    lines.append("═" * 78)

    # ADD THESE 4 LINES:
    lines.append("")
    lines.append("  ⚠️  MODEL NOTE: Trend model predicts next 20-day price direction.")
    lines.append("      Confidence <55% shows as 'Uncertain'. Current downtrends may")
    lines.append("      Model can show only uptrend or sideways. it is not trained for downtrend")
    
    # ── SECTION 1: TREND + REVERSAL TABLE ────────────────────
    lines.append("")
    lines.append("  TREND & REVERSAL SIGNALS")
    lines.append("  " + "─" * 74)
    lines.append(f"  {'Symbol':<12} {'Sector':<22} {'Trend':<12} {'Conf':>5}  {'Bottom%':>8} {'Top%':>6}")
    lines.append("  " + "─" * 74)

    for _, row in merged.iterrows():
        sym         = row['Symbol']
        sector      = str(row['Sector'])[:20]
        trend       = str(row.get('Trend_Label', 'N/A'))
        conf        = row.get('Trend_Conf', 0)
        bottom_prob = row.get('Bottom_Rev_Prob', 0)
        top_prob    = row.get('Top_Rev_Prob', 0)
        b_flag      = '⚡' if row.get('Bottom_Rev_Flag', False) else '  '
        t_flag      = '⚡' if row.get('Top_Rev_Flag', False) else '  '

        lines.append(
            f"  {sym:<12} {sector:<22} {trend:<12} {conf:>4.0f}%"
            f"  {b_flag}{bottom_prob:>5.1f}%  {t_flag}{top_prob:>4.1f}%"
        )

    # ── SECTION 2: 25D FORECAST ───────────────────────────────
    lines.append("")
    lines.append("  25-DAY FORECAST")
    lines.append("  " + "─" * 74)
    lines.append(f"  {'Symbol':<12} {'Current Price':>14} {'Expected':>10} {'Target Price':>13}")
    lines.append("  " + "─" * 74)

    for _, row in merged.iterrows():
        sym   = row['Symbol']
        price = row.get('Current Price', 0)
        pct   = row.get('Forecast_25d_Pct', None)
        tgt   = row.get('Forecast_25d_Price', None)
        if pct is None:
            continue
        sign = '+' if pct >= 0 else ''
        lines.append(
            f"  {sym:<12} {'Rs ' + str(round(price, 0)):>14}"
            f"  {sign}{pct:>7.1f}%  {'Rs ' + str(round(tgt, 0)):>12}"
        )

    # ── SECTION 3: 45D FORECAST ───────────────────────────────
    lines.append("")
    lines.append("  45-DAY FORECAST")
    lines.append("  " + "─" * 74)
    lines.append(f"  {'Symbol':<12} {'Current Price':>14} {'Expected':>10} {'Target Price':>13}")
    lines.append("  " + "─" * 74)

    for _, row in merged.iterrows():
        sym   = row['Symbol']
        price = row.get('Current Price', 0)
        pct   = row.get('Forecast_45d_Pct', None)
        tgt   = row.get('Forecast_45d_Price', None)
        if pct is None:
            continue
        sign = '+' if pct >= 0 else ''
        lines.append(
            f"  {sym:<12} {'Rs ' + str(round(price, 0)):>14}"
            f"  {sign}{pct:>7.1f}%  {'Rs ' + str(round(tgt, 0)):>12}"
        )

    # ── SECTION 4: 180D FORECAST ─────────────────────────────
    uptrend_stocks = merged[merged['Show_180d'] == True]

    if len(uptrend_stocks) > 0:
        lines.append("")
        lines.append("  6-MONTH FORECAST  (Uptrend confidence >60% only)")
        lines.append("  " + "─" * 74)
        lines.append(f"  {'Symbol':<12} {'Current Price':>14} {'Expected':>10} {'Target Price':>13}")
        lines.append("  " + "─" * 74)

        for _, row in uptrend_stocks.iterrows():
            sym   = row['Symbol']
            price = row.get('Current Price', 0)
            pct   = row.get('Forecast_180d_Pct', None)
            tgt   = row.get('Forecast_180d_Price', None)
            if pct is None:
                continue
            sign = '+' if pct >= 0 else ''
            lines.append(
                f"  {sym:<12} {'Rs ' + str(round(price, 0)):>14}"
                f"  {sign}{pct:>7.1f}%  {'Rs ' + str(round(tgt, 0)):>12}"
            )
    else:
        lines.append("")
        lines.append("  6-MONTH FORECAST  — No stocks qualify (need Uptrend + conf >60%)")

    lines.append("")
    lines.append("═" * 78)

    return "\n".join(lines)

# ── GENERATE AND PRINT ────────────────────────────────────────
tech_report = pd.read_csv('../data/technical_report.csv')
ml_report   = generate_ml_report(ml_scores_df, tech_report)
print(ml_report)

today_str   = date.today().strftime("%Y-%m-%d")
report_path = f'../data/ml_report_{today_str}.txt'
with open(report_path, 'w', encoding='utf-8') as f:
    f.write(ml_report)

print(f"\n✅ ML report saved: {report_path}")

══════════════════════════════════════════════════════════════════════════════
  ML SIGNALS — 22 March 2026
══════════════════════════════════════════════════════════════════════════════

  ⚠️  MODEL NOTE: Trend model predicts next 20-day price direction.
      Confidence <55% shows as 'Uncertain'. Current downtrends may
      Model can show only uptrend or sideways. it is not trained for downtrend

  TREND & REVERSAL SIGNALS
  ──────────────────────────────────────────────────────────────────────────
  Symbol       Sector                 Trend         Conf   Bottom%   Top%
  ──────────────────────────────────────────────────────────────────────────
  PERSISTENT   Information Technolo   Sideways       58%  ⚡ 64.7%     0.0%
  CHOLAFIN     Financial Services     Sideways       70%     49.8%     0.0%
  MUTHOOTFIN   Financial Services     Sideways       67%      0.2%     0.1%
  FINEORG      Chemicals              Sideways       74%     29.1%     0.1%
  TITAN        Consumer Durables      U